# Lesson 5

## Rebuilding GPT from scratch.

This time, we will build a GPT [generative pre-trained transformer](https://en.wikipedia.org/wiki/Generative_pre-trained_transformer) similar to Claude or ChatGPT, but of course, much smaller.

Our program is the following:

1. load and tokenize the dataset,
2. build a batch sampler,
3. train a **bigram** baseline,
4. understand the core idea of **causal self-attention**,
5. Replace the **bigram** to a **decoder-only Transformer**,
6. train it and generate text.

This class will bet different that the previous ones, as we do provide the code. The goal is not only to run the code, but to understand **why each component is there**.

In [1]:
# if running on google colab, run this cell
%cd /content
!rm -rf mini-course-DL
!git clone https://github.com/AdamArras/mini-course-DL.git
%cd mini-course-DL

/content
Cloning into 'mini-course-DL'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 72 (delta 24), reused 51 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 621.20 KiB | 2.96 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/mini-course-DL


# Part I : Preparations

## Imports and configuration

We start with a small set of hyperparameters.  
For notebook work, it is useful to have a quick configuration that runs fast, and a full configuration that we will run at the very end.

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

quick_run = False # True

if quick_run:
    batch_size = 32
    block_size = 128
    max_iters = 1000
    eval_interval = 200
    learning_rate = 3e-4
    eval_iters = 100
    n_embd = 192
    n_head = 6
    n_layer = 4
    dropout = 0.2
else:
    batch_size = 64
    block_size = 256
    max_iters = 5000
    eval_interval = 500
    learning_rate = 3e-4
    eval_iters = 200
    n_embd = 384
    n_head = 6
    n_layer = 6
    dropout = 0.2

device: cuda


## 1. Read the dataset

In [3]:
with open("data/shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("The dataset has " +str(len(text))+ " characters")
print("Here first 1000 characters:")
print("---------------------------")
print(text[:1000])

The dataset has 1115394 characters
Here first 1000 characters:
---------------------------
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we bec

## 2. Build the vocabulary

Because this is a **character-level** language model, the vocabulary is simply the sorted list of distinct characters appearing in the text.

As in the previous lessons, we need the follwing functions

- `stoi`: string-to-index
- `itos`: index-to-string
- `encode`: text -> list of integers
- `decode`: list of integers -> text

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("vocab size:", vocab_size)
print(chars)

vocab size: 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


## 3. Encode the whole corpus and split train / validation

- the first 90% for training,
- the last 10% for validation.

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("total tokens:", len(data))
print("train tokens:", len(train_data))
print("val tokens:", len(val_data))

total tokens: 1115394
train tokens: 1003854
val tokens: 111540


## 4. What is the training task?

For language modeling, we want to predict the next token. If the current chunk is:
`x = [t0, t1, t2, t3]`
then the targets are:
`y = [t1, t2, t3, t4]`
So every position tries to predict the next character.

In [6]:
example_block_size = 8
x = train_data[:example_block_size]
y = train_data[1:example_block_size + 1]

print("x:", x.tolist())
print("y:", y.tolist())
print()

for t in range(example_block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context.tolist()} the target is {target.item()}")

x: [18, 47, 56, 57, 58, 1, 15, 47]
y: [47, 56, 57, 58, 1, 15, 47, 58]

when input is [18] the target is 47
when input is [18, 47] the target is 56
when input is [18, 47, 56] the target is 57
when input is [18, 47, 56, 57] the target is 58
when input is [18, 47, 56, 57, 58] the target is 1
when input is [18, 47, 56, 57, 58, 1] the target is 15
when input is [18, 47, 56, 57, 58, 1, 15] the target is 47
when input is [18, 47, 56, 57, 58, 1, 15, 47] the target is 58


## 5. Batch sampling

Instead of always training on the same prefix, we sample many random chunks from the corpus. Each batch has shape:

- `x`: `(B, T)`
- `y`: `(B, T)`

where:

- `B = batch_size`
- `T = block_size`

In [7]:
def get_batch(split):
    data_source = train_data if split == "train" else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i + block_size] for i in ix])
    y = torch.stack([data_source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("xb shape:", xb.shape)
print("yb shape:", yb.shape)
print()
print("first training example x:")
print(xb[0])
print()
print("first training example decoded:")
print("---------------------------")
print(decode(xb[0].tolist())) # print(repr(decode(xb[0].tolist())))

xb shape: torch.Size([64, 256])
yb shape: torch.Size([64, 256])

first training example x:
tensor([ 0, 26, 53, 58,  1, 19, 50, 53, 59, 41, 43, 57, 58, 43, 56,  5, 57,  1,
        42, 43, 39, 58, 46,  6,  1, 52, 53, 56,  1, 20, 43, 56, 43, 44, 53, 56,
        42,  5, 57,  1, 40, 39, 52, 47, 57, 46, 51, 43, 52, 58,  0, 26, 53, 58,
         1, 19, 39, 59, 52, 58,  5, 57,  1, 56, 43, 40, 59, 49, 43, 57,  6,  1,
        52, 53, 56,  1, 17, 52, 45, 50, 39, 52, 42,  5, 57,  1, 54, 56, 47, 60,
        39, 58, 43,  1, 61, 56, 53, 52, 45, 57,  6,  0, 26, 53, 56,  1, 58, 46,
        43,  1, 54, 56, 43, 60, 43, 52, 58, 47, 53, 52,  1, 53, 44,  1, 54, 53,
        53, 56,  1, 14, 53, 50, 47, 52, 45, 40, 56, 53, 49, 43,  0, 13, 40, 53,
        59, 58,  1, 46, 47, 57,  1, 51, 39, 56, 56, 47, 39, 45, 43,  6,  1, 52,
        53, 56,  1, 51, 63,  1, 53, 61, 52,  1, 42, 47, 57, 45, 56, 39, 41, 43,
         6,  0, 20, 39, 60, 43,  1, 43, 60, 43, 56,  1, 51, 39, 42, 43,  1, 51,
        43,  1, 57, 53, 59, 5

## 6. Loss estimation helper

During training we periodically average the loss over several batches for both train and validation splits.

In [8]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Part II : The bigram baseline

We first consider the bigram model, which predicts the next character using only the current character. This model is not very powerful, but it is the first step toward a true transformer architecture.

## 7. Bigram language model

The entire model is just an embedding table of shape `(vocab_size, vocab_size)`.

For each current token, it directly stores logits for the next token.

In [9]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]           # (B, C)
            probs = F.softmax(logits, dim=-1)   # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)             # (B, T+1)
        return idx

## 8. Sanity check the bigram model

Before training, the loss should be close to `log(vocab_size)`, because predictions are almost random.

In [10]:
bigram_model = BigramLanguageModel(vocab_size).to(device)
logits, loss = bigram_model(xb, yb)

print("log(vocab_size) = ", np.log(vocab_size))
print("logits shape:", logits.shape)
print("initial loss:", loss.item())

log(vocab_size) =  4.174387269895637
logits shape: torch.Size([64, 256, 65])
initial loss: 4.739931106567383


## 9. Train the bigram model

In [11]:
optimizer = torch.optim.AdamW(bigram_model.parameters(), lr=1e-2)

for step in range(1000):
    if step % 200 == 0:
        losses = estimate_loss(bigram_model)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = bigram_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.7524, val loss 4.7588
step 200: train loss 2.9275, val loss 2.9447
step 400: train loss 2.5589, val loss 2.5822
step 600: train loss 2.4986, val loss 2.5237
step 800: train loss 2.4814, val loss 2.5076


## 10. Generate text with the bigram model

In [12]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
sample = bigram_model.generate(context, max_new_tokens=400)[0].tolist()
print(decode(sample))




CEThik brid owindakis by bth

HAPet bube d e.
S:
O:3 my d?
LUCous:
Wanthar u qur, vet?
F dXENDoate awice my.

Hastarom oroup
Yowhthetof isth ble mil ndill, ath iree sengmin lat Heriliovets, and Win nghir.
Thanousel lind me l.
HAshe ce hiry:
Supr aisspllw y.
HerinUSI y mopetelaves
MP:

Pl, d mothakleo Windo whthCoribyo the m dourive we higend t so mower; te

AN ad nterupt f s ar iris! m:

Thiny a


The text is usually noisy and short-sighted, which is exactly what we should expect from a model with no real context.

# Part III : Understanding self-attention

The bigram model has no memory beyond the current token.

A Transformer fixes this by letting each position build a representation from the **previous positions in the sequence**.

We now isolate the main idea behind attention.

## 11. A toy example: averaging the past with a lower-triangular mask

Suppose each time step wants access to the average of all previous steps, including itself.

A causal mask is lower triangular because position `t` can only look at positions `<= t`.

In [13]:
torch.manual_seed(42)

B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

wei = torch.tril(torch.ones(T, T)) # wei is short for weight
wei = wei / wei.sum(1, keepdim=True)
out = wei @ x

print("x shape:", x.shape)
print("wei shape:", wei.shape)
print("out shape:", out.shape)
print()
print("first example, first channel:")
print("input :", x[0, :, 0])
print("output:", out[0, :, 0])

x shape: torch.Size([4, 8, 2])
wei shape: torch.Size([8, 8])
out shape: torch.Size([4, 8, 2])

first example, first channel:
input : tensor([ 1.9269,  0.9007,  0.6784, -0.0431, -0.7521, -0.3925, -0.7279, -0.7688])
output: tensor([1.9269, 1.4138, 1.1687, 0.8657, 0.5422, 0.3864, 0.2272, 0.1027])


This is already a form of communication across time, but it is very limited:

- every previous token gets a fixed weight,
- the weights do not depend on the content of the sequence.

Attention improves this by making the weights **data-dependent**.

## 12. From fixed averaging to learned attention weights

The key idea is:

- build a **query** vector from each token,
- build a **key** vector from each token,
- compare queries and keys with a dot product,
- normalize with softmax,
- use the resulting weights to mix the **value** vectors.

In [14]:
torch.manual_seed(42)

B, T, C = 4, 8, 32
head_size = 16

x = torch.randn(B, T, C)
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, hs)
q = query(x)    # (B, T, hs)
v = value(x)    # (B, T, hs)

wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

out = wei @ v

print("k shape:", k.shape)
print("q shape:", q.shape)
print("attention weights shape:", wei.shape)
print("output shape:", out.shape)

k shape: torch.Size([4, 8, 16])
q shape: torch.Size([4, 8, 16])
attention weights shape: torch.Size([4, 8, 8])
output shape: torch.Size([4, 8, 16])


# Part IV — Building the Transformer

We now implement the same GPT architecture as in the final script:

- token embeddings,
- positional embeddings,
- causal self-attention,
- feedforward layers,
- residual connections,
- layer normalization,
- final language-model head.

## 14. One attention head

In [15]:
class Head(nn.Module):
    """One head of causal self-attention."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)                         # (B, T, hs)
        q = self.query(x)                       # (B, T, hs)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)                       # (B, T, hs)
        out = wei @ v                           # (B, T, hs)
        return out

## 15. Multi-head attention

Instead of using a single attention map, we use several heads in parallel.  
Each head can specialize in a different type of relation.

In [16]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 16. Feedforward network

After tokens communicate through attention, each position is processed independently through a small MLP.

In [17]:
class FeedForward(nn.Module):
    """A position-wise MLP."""

    def __init__(self, n_embd):
        super().__init__()
        dim_ff = 4* n_embd # common choice of the dim of the feed foward layer, as in the original transformer paper
        self.net = nn.Sequential(
            nn.Linear(n_embd, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## 17. Transformer block

A block alternates:

1. **communication** via self-attention,
2. **computation** via the feedforward network.

We also use:

- **residual connections**,
- **pre-layer normalization**.

In [18]:
class Block(nn.Module):
    """Transformer block: communication followed by computation."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 18. Full GPT language model

This is a **decoder-only Transformer**:

- token embeddings identify the characters,
- positional embeddings encode the time index,
- a stack of Transformer blocks mixes information causally,
- a final linear layer produces logits over the vocabulary.

In [19]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head=n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                             # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb                                                 # (B, T, C)

        x = self.blocks(x)                                                    # (B, T, C)
        x = self.ln_f(x)                                                      # (B, T, C)
        logits = self.lm_head(x)                                              # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 19. Initialisation of the model

In [20]:
model = GPTLanguageModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.3f} M parameters")

10.789 M parameters


## 20. Check one forward pass

In [21]:
xb, yb = get_batch("train")
logits, loss = model(xb, yb)

print("logits shape:", logits.shape)
print("loss:", loss.item())

logits shape: torch.Size([16384, 65])
loss: 4.235957622528076


## 21. Train the Transformer

In [22]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2352, val loss 4.2341
step 500: train loss 1.7456, val loss 1.9103
step 1000: train loss 1.3973, val loss 1.6090
step 1500: train loss 1.2727, val loss 1.5312
step 2000: train loss 1.1930, val loss 1.5005
step 2500: train loss 1.1280, val loss 1.4822
step 3000: train loss 1.0697, val loss 1.4894
step 3500: train loss 1.0180, val loss 1.4922
step 4000: train loss 0.9649, val loss 1.4992
step 4500: train loss 0.9129, val loss 1.5330
step 4999: train loss 0.8615, val loss 1.5568


## 22. Generate from the trained GPT

In [23]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
sample = model.generate(context, max_new_tokens=600)[0].tolist()
print(decode(sample))


gave you, father; which I may bite my forward;
'Touchides not be already, 'lay not me long.

JULIET:
If she you'll be no friend royalty, provoked by this:
O loyalter mother, of my wife to blow
Leave me farewell throw out some other summore.
Saw you tell me her o'erlasted when want together
Aharmented 'twas any harlight and to do thee cut
That I can no mean but state writtle.

ESCALUS:
Ay that which find will forget it to encounter.

POMPEY:
Bull, who would be traither but with one can discover pass
love.

JUSHN:
Go, bind them before their cares and tops them,
Even for bravenom her compets. All
